In [0]:
#Read Silver Data
silver_finance_df = spark.table("silver_finance")

In [0]:
#Create High-Value Transaction Flag
from pyspark.sql.functions import when, col

fraud_df = silver_finance_df.withColumn(
    "high_value_flag",
    when(col("transaction_amount") > 4000, 1)
    .otherwise(0)
)

In [0]:
#Risky Payment Method Flag
fraud_df = fraud_df.withColumn(
    "risky_payment_flag",
    when(col("transaction_type") == "Credit Card", 1)
    .otherwise(0)
)

In [0]:
#Build Risk Score
from pyspark.sql.functions import expr

fraud_df = fraud_df.withColumn(
    "risk_score",
    expr("""
        (high_value_flag * 70)
        +
        (risky_payment_flag * 30)
    """)
)

In [0]:
#Create Risk Category
fraud_df = fraud_df.withColumn(
    "risk_category",
    when(col("risk_score") >= 70, "HIGH")
    .when(col("risk_score") >= 30, "MEDIUM")
    .otherwise("LOW")
)

In [0]:
#Verify Results
display(
    fraud_df.select(
        "transaction_id",
        "transaction_amount",
        "transaction_type",
        "risk_score",
        "risk_category"
    )
)

In [0]:
high_risk_df=fraud_df.filter(
    col("risk_category") == "HIGH"
    )


In [0]:
#Save Fraud Transactions
high_risk_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_high_risk_transactions")

In [0]:
#Aggregate risk by customer
from pyspark.sql.functions import avg,count

customer_risk_df=fraud_df.groupBy("customer_id"
                                 
).agg(
    avg("risk_score").alias("avg_risk_score"),
    count("*").alias("transaction_count")

)

In [0]:
#Risk Category for Customers
customer_risk_df = customer_risk_df.withColumn(
    "customer_risk_level",
    when(col("avg_risk_score") >= 60, "HIGH")
    .when(col("avg_risk_score") >= 30, "MEDIUM")
    .otherwise("LOW")
)

In [0]:
customer_risk_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_customer_risk")

In [0]:
#Fraud Dashboard Summary
fraud_dashboard_df = fraud_df.groupBy(
    "risk_category"
).count()

In [0]:
#Save Dashboard Table
fraud_dashboard_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_fraud_dashboard")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM gold_fraud_dashboard
    """)
)